# Fase CRISP-DM: Gestión de Datos Maestros (MDM) y Resolución de Entidades
**Plataforma**: AgroData Intelligence Platform (AgroStatsApp)  
**Estándares**: DAMA-DMBOK 2 (Master & Reference Data), DANE DIVIPOLA, Clasificación CPC Ver. 2.1 A.C.  
**Propósito**: Estandarizar identificadores de departamentos (DIVIPOLA), clasificar productos agropecuarios según nomenclatura CPC, extraer catálogos canónicos de estaciones meteorológicas y plazas mayoristas, y generar claves surrogadas para la fusión cross-domain.

In [ ]:
# 1. Configuración de Entorno Resiliente (Google Colab / VS Code / Jupyter Local)
import os
import sys
import subprocess
from pathlib import Path

def setup_environment():
    # A. Detección y preparación automática para Google Colab
    if 'google.colab' in sys.modules or Path('/content').exists():
        print('[INFO] Entorno detectado: Google Colab.')
        repo_dir = Path('/content/Statsfirm')
        if not repo_dir.exists():
            print('[INFO] Clonando repositorio oficial Statsfirm en Colab...')
            subprocess.run(['git', 'clone', 'https://github.com/adansanchezc1-spec/Statsfirm.git', '/content/Statsfirm'], check=True)
        else:
            print('[INFO] Actualizando repositorio en Colab...')
            subprocess.run(['git', '-C', '/content/Statsfirm', 'pull'], check=False)
        
        app_dir = repo_dir / 'AgroStats AndTech' / 'AgroStatsApp'
        if app_dir.exists():
            os.chdir(str(app_dir))
            src_dir = app_dir / 'src'
            if str(src_dir) not in sys.path:
                sys.path.insert(0, str(src_dir))
            print(f'[OK] Directorio de trabajo establecido en: {app_dir}')
            print(f'[OK] Carpeta src agregada a sys.path: {src_dir}')
            return

    # B. Detección dinámica en Entorno Local (Windows / Linux / WSL / VS Code)
    candidates = [
        Path.cwd() / 'src',
        Path.cwd() / 'AgroStats AndTech' / 'AgroStatsApp' / 'src',
        Path.cwd().parent / 'src',
        Path.cwd().parent.parent / 'src',
        Path.cwd().parent.parent / 'AgroStats AndTech' / 'AgroStatsApp' / 'src',
        Path(r'c:\Users\ADAN\OneDrive\Documentos\Statsfirm\AgroStats AndTech\AgroStatsApp\src'),
        Path('/mnt/c/Users/ADAN/OneDrive/Documentos/Statsfirm/AgroStats AndTech/AgroStatsApp/src'),
    ]
    
    curr = Path.cwd().resolve()
    for _ in range(6):
        target = curr / 'AgroStats AndTech' / 'AgroStatsApp' / 'src'
        if target.exists() and (target / 'notebook_code').is_dir():
            candidates.insert(0, target)
            break
        curr = curr.parent

    for c in candidates:
        if c.exists() and (c / 'notebook_code').is_dir():
            resolved = str(c.resolve())
            if resolved not in sys.path:
                sys.path.insert(0, resolved)
            print(f'[OK] Módulo src localizado localmente en: {resolved}')
            return

setup_environment()

# Importación del gestor de datos maestros
from notebook_code import MasterDataManager, load_all_raw_datasets

print('[OK] Módulo MasterDataManager importado exitosamente.')


In [ ]:
# 2. Carga de Datos Crudos para Armonización
datasets = load_all_raw_datasets()
df_precios = datasets['sipsa_precios']
df_pluvio = datasets['ideam_pluvio']
print('Datasets cargados para gobierno de datos maestros.')


In [ ]:
# 3. Armonización DANE DIVIPOLA Departamental
dept_col = next((c for c in df_pluvio.columns if 'departamento' in c.lower()), 'departamento')
df_divipola = MasterDataManager.harmonize_divipola(df_pluvio, dept_col=dept_col)
print('Muestra de asignación DIVIPOLA:')
display(df_divipola[[dept_col, 'depto_normalizado', 'cod_dpto_divipola']].drop_duplicates().head(10))


In [ ]:
# 4. Armonización CPC de Productos Agropecuarios (SIPSA)
prod_col = next((c for c in df_precios.columns if 'articulo' in c or 'producto' in c), df_precios.columns[0])
df_cpc = MasterDataManager.harmonize_cpc_products(df_precios, product_col=prod_col)
print('Muestra de clasificación CPC:')
display(df_cpc[[prod_col, 'producto_normalizado', 'cpc_code', 'cpc_nombre', 'cpc_grupo']].drop_duplicates().head(10))


In [ ]:
# 5. Extracción de Catálogo Maestro de Estaciones Meteorológicas IDEAM
cat_estaciones = MasterDataManager.extract_ideam_stations_catalog(df_pluvio)
print(f'Total de estaciones identificadas: {len(cat_estaciones)}')
display(cat_estaciones.head(10))


In [ ]:
# 6. Extracción de Catálogo Maestro de Plazas Mayoristas SIPSA
cat_mercados = MasterDataManager.extract_sipsa_markets_catalog(df_precios)
print(f'Total de mercados identificados: {len(cat_mercados)}')
display(cat_mercados.head(10))


In [ ]:
# 7. Generación de Claves Surrogadas Geo-Temporales para Integración
df_con_llave = MasterDataManager.build_cross_domain_surrogate_key(
    df_pluvio,
    date_col='fechaobservacion',
    dept_col='departamento'
)
print('Muestra de claves surrogadas:')
display(df_con_llave[['fechaobservacion', 'departamento', 'cod_dpto_divipola', 'surrogate_key_geo_temporal']].head(10))


In [ ]:
# 8. Persistencia de Catálogos Maestros en CRISPDM/data/MDM/
rutas_mdm = MasterDataManager.save_mdm_catalogs({
    'estaciones_ideam': cat_estaciones,
    'mercados_sipsa': cat_mercados,
})
print('Catálogos persistidos exitosamente:')
for k, v in rutas_mdm.items():
    print(f'• {k}: {v}')
